In [4]:
pip install statsmodels --user


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# -*- coding: utf-8 -*-
"""
King County EV Adoption & Charger Forecast — Monotonic Monte Carlo
------------------------------------------------------------------
✔ Preserves original logic, outputs, 3 plots, and Excel embedding
✔ Adds a single parameters block for all decimal constants
✔ Adds concise inline explanations where decimals are used
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from openpyxl import load_workbook
from openpyxl.drawing.image import Image
import matplotlib.patches as mpatches
from matplotlib.ticker import PercentFormatter

# =============================================================================
# MODEL PARAMETERS (all decimal constants centralized with descriptions)
# =============================================================================

# --- Policy adoption targets (share of population 25–59 that adopt EV) ---
POLICY_TARGETS = {2030: 0.45, 2035: 0.60, 2040: 0.70, 2050: 0.95}  # e.g., 0.45 = 45% target by 2030

# --- Geo/pop charger targeting ---
RADIUS_MILES = 2.0             # 2-mile service radius per charger (geo coverage calc)
POP_PER_CHARGER = 1500.0       # Upper-bound target: 1 public charger per 1,500 residents

# --- Early adoption baseline (used in Lower & Upper scenarios) ---
BASE_2025_ADOPTION = 0.10      # 10% baseline adoption in 2025

# --- Lower-scenario tuning ---
LOWER_P1_MOD = 0.35            # Use 35% of early-growth toward 2030 target in Phase 1 (moderation)
LOWER_P2_MID = 0.70            # Use 70% of 2040 policy target as Phase 2 midpoint plateau

# --- Upper-scenario build distribution (front-load adds, then taper) ---
UP_PHASE1_W = 1.0              # 2025–2030 weight for adds
UP_PHASE2_W = 0.6              # 2031–2040 weight for adds
UP_PHASE3_W = 0.2              # 2041–2050 weight for adds

# --- Monte Carlo: adoption cap / targets blending ---
TARGET_CAP = 0.94              # Keep MC under ~95% by 2050 to avoid hitting 100%
W_UPPER_P1 = 0.55              # 55% weight to Upper target in 2025–2030
W_UPPER_P2 = 0.25              # 25% weight to Upper target in 2031–2035 (lean low)
W_UPPER_P3 = 0.70              # 70% weight to Upper target from 2036 onward (accelerate)

# --- Charger blending weights (building the budget-weighted monotone charger path) ---
W_CHARGER_P1 = 0.80            # phase 1 blend with Upper chargers (strongly upper-leaning early)
W_CHARGER_P2 = 0.30            # phase 2 blend (home-charger transition reduces public need)
W_CHARGER_P3 = 0.55            # phase 3 blend (catch-up toward higher public share again)

# --- Home charger share over time (dampens effect of public network growth) ---
HOME_P1_START = 0.55           # 55% of charging at home at start (2025)
HOME_P1_END   = 0.72           # rises to 72% by 2030
HOME_P2_END   = 0.90           # reaches 90% by 2040
HOME_P3_END   = 0.93           # nudges to 93% by 2050

# --- Charger growth rate handling ---
TRAIL_YEARS = 3                # trailing window for growth momentum average
MOMENTUM_CAP = 0.40            # cap charger momentum factor to 40%
GROWTH_RATE_CAP = 0.50         # cap annual charger growth rate to 50%

# --- Monte Carlo behavior by phase ---
# Phases: P1=2025–2030, P2=2031–2035, P3=2036–2040, P4=2041–2050
MC_KAPPA = {"P1": 0.30, "P2": 0.25, "P3": 0.45, "P4": 0.55}  # mean-reversion to target
MC_BETA  = {"P1": 0.45, "P2": 0.20, "P3": 0.30, "P4": 0.30}  # charger-growth coupling baseline
MC_SIGMA = {"P1": 0.015, "P2": 0.010, "P3": 0.018, "P4": 0.020}  # noise std by phase
MC_BETA_MOMENTUM_MULT = 1.5   # amplify charger momentum on beta (1 + 1.5*momentum)
MC_BETA_MAX = 0.9             # max effective beta to keep stability
MC_START_BLEND_LO = 0.35       # MC start level: 35% weight to lower bound rate at t0
MC_START_BLEND_HI = 0.65       # MC start level: 65% weight to upper bound rate at t0
MC_RUNS = 1000                 # number of Monte Carlo paths

# --- Chart phase visualization ---
PHASE_SPANS = [(2025, 2030, "Phase 1"), (2030, 2040, "Phase 2"), (2040, 2050, "Phase 3")]
PHASE_COLORS = ["green", "yellow", "orange"]  # visualization only

# =============================================================================
# Paths (adjust if needed)
# =============================================================================
super_path = "/Users/judycheng/Desktop/supercharger in washington state.xls"
residents_path = "/Users/judycheng/Desktop/Population 2024 age 25 to 59.xlsx"
ev_path = "/Users/judycheng/Desktop/coordinates_output.xlsm"

# =============================================================================
# Step 1: Read Excel files
# =============================================================================
super_df = pd.read_excel(super_path)
residents_df = pd.read_excel(residents_path)
ev_df = pd.read_excel(ev_path)

# =============================================================================
# Step 2: Filter to King County data
# =============================================================================
king_super = super_df[(super_df["State"] == "Washington") & (super_df["County"] == "King County")]
king_residents = residents_df[residents_df["County"] == "King County"]
king_ev = ev_df[(ev_df["State"] == "WA") & (ev_df["County"] == "King")]

population_column = "total"
population_25_59 = float(king_residents[population_column].values[0])
num_chargers = int(len(king_super))
num_evs = int(len(king_ev))

print(f"King County population (25-59): {population_25_59:,.0f}")
print(f"Existing Superchargers: {num_chargers}")
print(f"Existing EVs (rows in EV file for King): {num_evs}")

# =============================================================================
# Step 3: Policy targets
# =============================================================================
policy_targets = POLICY_TARGETS  # keep original dict name used below

# =============================================================================
# Step 4: Charger target math
# =============================================================================
king_county_area = 2307  # sq mi (given)
area_per_charger = np.pi * RADIUS_MILES**2  # π * r^2 using RADIUS_MILES=2.0
lower_chargers_goal = int(np.ceil(king_county_area / area_per_charger))      # geo coverage (2-mile)
upper_chargers_goal = int(round(population_25_59 / POP_PER_CHARGER))         # 1 per 1,500 residents

print(f"Lower bound chargers (geo coverage): {lower_chargers_goal}")
print(f"Upper bound chargers (1 per 1,500 residents): {upper_chargers_goal}")

# =============================================================================
# Step 5: Build scenarios
# =============================================================================
years = list(range(2025, 2051))
n_years = len(years)
rows = []

# ---- Lower: slow Phase 2, catch-up Phase 3 ----
current_lower = num_chargers
total_lower_needed = max(0, lower_chargers_goal - num_chargers)
yearly_build_lower = int(round(total_lower_needed / n_years)) if n_years > 0 else 0

for y in years:
    # Chargers build (constant toward geo cap)
    new_lower = yearly_build_lower
    if current_lower + new_lower > lower_chargers_goal:
        new_lower = lower_chargers_goal - current_lower
    current_lower += max(0, new_lower)

    # Adoption: P1 modest; P2 slower midpoint; P3 catch-up
    if y <= 2030:
        # BASE_2025_ADOPTION + (gap to 2030 target) * progress * LOWER_P1_MOD
        eff = BASE_2025_ADOPTION + (policy_targets[2030] - BASE_2025_ADOPTION) * ((y - 2025) / 5) * LOWER_P1_MOD
    elif y <= 2040:
        eff_start = BASE_2025_ADOPTION + (policy_targets[2030] - BASE_2025_ADOPTION) * LOWER_P1_MOD
        # drift toward 70% of 2040 target by 2040 (LOWER_P2_MID)
        eff = eff_start + (policy_targets[2040] * LOWER_P2_MID - eff_start) * ((y - 2030) / 10)
    else:
        eff_start = policy_targets[2040] * LOWER_P2_MID
        # catch-up to 2050 target from 2040–2050
        eff = eff_start + (policy_targets[2050] - eff_start) * ((y - 2040) / 10)

    rows.append({
        "Scenario": "Lower",
        "Year": y,
        "Total_Chargers": int(current_lower),
        "New_Chargers": int(max(0, new_lower)),
        "Adoption_Rate": float(np.clip(eff, 0.0, 0.9999))  # keep <100%
    })

# ---- Upper: 3-phase slowdown, aggressive to meet targets ----
current_upper = num_chargers
total_upper_needed = max(0, upper_chargers_goal - num_chargers)

# Allocate adds by phase (front-load then taper)
phase_weights = []
for y in years:
    if 2025 <= y <= 2030:
        phase_weights.append(UP_PHASE1_W)
    elif 2031 <= y <= 2040:
        phase_weights.append(UP_PHASE2_W)
    else:
        phase_weights.append(UP_PHASE3_W)

phase_weights = np.array(phase_weights)
phase_weights = phase_weights / phase_weights.sum()
yearly_adds = np.round(phase_weights * total_upper_needed).astype(int)

# Ensure exact sum
diff = total_upper_needed - yearly_adds.sum()
if diff != 0:
    idx = 0
    step = 1 if diff > 0 else -1
    for _ in range(abs(diff)):
        yearly_adds[idx] += step
        idx = (idx + 1) % len(yearly_adds)

for y, add in zip(years, yearly_adds):
    current_upper += max(0, int(add))
    # Piecewise adoption path that hits 2030, 2035, 2040, 2050 policy points
    if y <= 2030:
        eff = BASE_2025_ADOPTION + (policy_targets[2030] - BASE_2025_ADOPTION) * ((y - 2025) / 5)
    elif y <= 2035:
        eff = policy_targets[2030] + (policy_targets[2035] - policy_targets[2030]) * ((y - 2030) / 5)
    elif y <= 2040:
        eff = policy_targets[2035] + (policy_targets[2040] - policy_targets[2035]) * ((y - 2035) / 5)
    else:
        eff = policy_targets[2040] + (policy_targets[2050] - policy_targets[2040]) * ((y - 2040) / 10)

    rows.append({
        "Scenario": "Upper",
        "Year": y,
        "Total_Chargers": int(current_upper),
        "New_Chargers": int(max(0, int(add))),
        "Adoption_Rate": float(np.clip(eff, 0.0, 0.9999))
    })

sc_df = pd.DataFrame(rows)

# =============================================================================
# Step 6: Monte Carlo (Monotone; adoption responds to charger buildup)
# =============================================================================
yrs = sc_df["Year"].unique()
lo = sc_df.pivot(index="Year", columns="Scenario", values="Adoption_Rate")["Lower"].values
hi = sc_df.pivot(index="Year", columns="Scenario", values="Adoption_Rate")["Upper"].values

# --- 6A) Target path the MC reverts toward (NOT midpoint) ---
# phase-weighted blend between lower/upper adoption paths
target_w_upper = np.array([
    (W_UPPER_P1 if 2025 <= y <= 2030 else (W_UPPER_P2 if 2031 <= y <= 2035 else W_UPPER_P3))
    for y in yrs
], dtype=float)

raw_target = (1 - target_w_upper) * lo + target_w_upper * hi
target = np.minimum(raw_target, TARGET_CAP)     # cap at 94%
target = np.maximum.accumulate(target)          # ensure target non-decreasing

# --- 6B) Public charger growth & home-charger dampening ---
w_upper_charger = np.array([
    (W_CHARGER_P1 if 2025 <= y <= 2030 else (W_CHARGER_P2 if 2031 <= y <= 2040 else W_CHARGER_P3))
    for y in yrs
], dtype=float)

lowC = sc_df.pivot(index="Year", columns="Scenario", values="Total_Chargers")["Lower"].values
hiC  = sc_df.pivot(index="Year", columns="Scenario", values="Total_Chargers")["Upper"].values

# Blended charger path then enforce monotonicity
C_blend = ((1 - w_upper_charger) * lowC + w_upper_charger * hiC).round().astype(int)
C_blend = np.maximum.accumulate(C_blend)        # monotonic chargers
assert np.all(np.diff(C_blend) >= 0), "Internal: blended chargers decreased unexpectedly."

# Convert to growth metrics
C_growth = np.r_[0, np.diff(C_blend)]
den = np.maximum(1.0, np.r_[C_blend[0], C_blend[:-1]])
C_growth_rate = np.divide(C_growth, den, out=np.zeros_like(C_growth, dtype=float), where=den > 0)
C_growth_rate = np.clip(C_growth_rate, 0, GROWTH_RATE_CAP)  # cap growth rate

# Home-charger share trajectory (dampens public network marginal effect)
home_share = []
for y in yrs:
    if 2025 <= y <= 2030:
        # linear from HOME_P1_START to HOME_P1_END over 5 years
        home_share.append(HOME_P1_START + (HOME_P1_END - HOME_P1_START) * (y - 2025) / 5)
    elif 2031 <= y <= 2040:
        # linear from 2030->2040 HOME_P1_END->HOME_P2_END
        home_share.append(HOME_P1_END + (HOME_P2_END - HOME_P1_END) * (y - 2030) / 10)
    else:
        # linear from 2040->2050 HOME_P2_END->HOME_P3_END
        home_share.append(HOME_P2_END + (HOME_P3_END - HOME_P2_END) * (y - 2040) / 10)
home_share = np.array(home_share, dtype=float)

# Momentum of charger build-out (simple trailing avg of growth rate)
pad = np.full(TRAIL_YEARS - 1, C_growth_rate[1])  # repeat first nonzero-ish growth
C_gr_trailing = np.convolve(np.r_[pad, C_growth_rate], np.ones(TRAIL_YEARS) / TRAIL_YEARS, mode="valid")
C_gr_trailing = C_gr_trailing[:len(C_growth_rate)]
charger_momentum = np.clip(C_gr_trailing, 0.0, MOMENTUM_CAP)  # cap

# ======================
# 6C) Monotone MC in rate space (adoption responds to growth & momentum)
# ======================
N = MC_RUNS
T = len(yrs)
rng = np.random.default_rng(42)

# Phase parameters
kappa_base = np.zeros(T)
beta_base  = np.zeros(T)
sigma      = np.zeros(T)
for i, y in enumerate(yrs):
    if 2025 <= y <= 2030:
        kappa_base[i] = MC_KAPPA["P1"]; beta_base[i] = MC_BETA["P1"]; sigma[i] = MC_SIGMA["P1"]
    elif 2031 <= y <= 2035:
        kappa_base[i] = MC_KAPPA["P2"]; beta_base[i] = MC_BETA["P2"]; sigma[i] = MC_SIGMA["P2"]
    elif 2036 <= y <= 2040:
        kappa_base[i] = MC_KAPPA["P3"]; beta_base[i] = MC_BETA["P3"]; sigma[i] = MC_SIGMA["P3"]
    else:
        kappa_base[i] = MC_KAPPA["P4"]; beta_base[i] = MC_BETA["P4"]; sigma[i] = MC_SIGMA["P4"]

# Adaptive coupling to chargers (beta), with home-share dampening and momentum amplification
beta_effect = beta_base * (1 - home_share) * (1 + MC_BETA_MOMENTUM_MULT * charger_momentum)
beta_effect = np.clip(beta_effect, 0.0, MC_BETA_MAX)

# Initialize paths
paths = np.zeros((N, T), dtype=float)
start_level = float(np.clip(MC_START_BLEND_LO * lo[0] + MC_START_BLEND_HI * hi[0], 0.0, TARGET_CAP))
paths[:, 0] = start_level

for t in range(1, T):
    prev = paths[:, t - 1]
    mean_revert  = kappa_base[t] * (target[t] - prev)  # pulls up to target if below
    charger_push = beta_effect[t] * C_growth_rate[t]   # >= 0, more when network expands
    noise        = rng.normal(0.0, sigma[t], size=N)   # small, zero-mean
    delta = mean_revert + charger_push + noise
    delta = np.maximum(delta, 0.0)                     # forbid negative increments (monotone)
    next_rate = prev + delta
    next_rate = np.maximum(next_rate, lo[t])           # never below Lower for that year
    next_rate = np.minimum(next_rate, TARGET_CAP)      # keep under cap
    next_rate = np.maximum(next_rate, prev)            # enforce monotonicity
    paths[:, t] = next_rate

# Percentiles per year
p10 = np.percentile(paths, 10, axis=0)
p50 = np.percentile(paths, 50, axis=0)
p90 = np.percentile(paths, 90, axis=0)

ev_p10 = (p10 * population_25_59).astype(int)
ev_p50 = (p50 * population_25_59).astype(int)
ev_p90 = (p90 * population_25_59).astype(int)

# =============================================================================
# Step 7: Build final table with new columns
# =============================================================================
wide = sc_df.pivot(index="Year", columns="Scenario", values=["Total_Chargers", "Adoption_Rate"])
wide.columns = [f"{a}_{b}" for a, b in wide.columns]
wide = wide.reset_index()

# Budget-weighted charger path as Forecast_Chargers (already monotone)
budget_weighted_C = C_blend.copy()
assert np.all(np.diff(budget_weighted_C) >= 0), "Forecast chargers decreased — check weighting!"
wide["Forecast_Chargers"] = budget_weighted_C

# MC outputs
wide["Forecast_Adoption_P10"] = p10
wide["Forecast_Adoption_P50"] = p50
wide["Forecast_Adoption_P90"] = p90
wide["Forecast_EVs_P10"] = ev_p10
wide["Forecast_EVs_P50"] = ev_p50
wide["Forecast_EVs_P90"] = ev_p90

# Optional check: how often P50 sits within [Lower, Upper]
within_band = (
    (wide["Forecast_Adoption_P50"] >= wide["Adoption_Rate_Lower"])
    & (wide["Forecast_Adoption_P50"] <= wide["Adoption_Rate_Upper"])
)
wide["Forecast_within_bounds"] = within_band.astype(int)

# =============================================================================
# Step 8: Save results to Excel
# =============================================================================
desktop_path = os.path.join(os.path.expanduser("~"), "Desktop", "king_county_ev_projection_mc_monotonic.xlsx")
with pd.ExcelWriter(desktop_path, engine="openpyxl") as writer:
    sc_df.to_excel(writer, sheet_name="Scenarios_LU", index=False)
    wide.to_excel(writer, sheet_name="Forecast", index=False)

# =============================================================================
# Step 9: Charts (with phases) — PRESERVED
# =============================================================================
phase_spans = PHASE_SPANS
phase_colors = PHASE_COLORS

# --- A) Total Chargers: Lower vs Upper vs Budget-weighted Forecast ---
plt.figure(figsize=(10, 6))
ax1 = plt.gca()
ax1.plot(wide["Year"], wide["Total_Chargers_Lower"], marker="o", label="Lower Bound")
ax1.plot(wide["Year"], wide["Total_Chargers_Upper"], marker="o", label="Upper Bound")
ax1.plot(wide["Year"], wide["Forecast_Chargers"], linestyle="--", linewidth=2, label="Budget-Weighted Forecast (Monotone)")
for (start, end, name), c in zip(phase_spans, phase_colors):
    ax1.axvspan(start, end, color=c, alpha=0.10)
line_handles, line_labels = ax1.get_legend_handles_labels()
phase_handles = [mpatches.Patch(color=c, alpha=0.10, label=name) for c, (_, _, name) in zip(phase_colors, phase_spans)]
ax1.legend(line_handles + phase_handles, line_labels + [name for *_, name in phase_spans], loc="best")
ax1.set_xlabel("Year")
ax1.set_ylabel("Total Chargers")
ax1.set_title("King County: EV Chargers — Lower vs Upper vs Budget-Weighted (Monotone)")
ax1.grid(True)
plt.tight_layout()
charger_chart = os.path.join(os.path.expanduser("~"), "Desktop", "charger_projection_mc_monotonic.png")
plt.savefig(charger_chart)
plt.close()

# --- B) EV Adoption Rate: Lower vs Upper vs Monotonic MC ---
plt.figure(figsize=(10, 6))
ax2 = plt.gca()
ax2.plot(wide["Year"], wide["Adoption_Rate_Lower"], marker="o", label="Lower Bound")
ax2.plot(wide["Year"], wide["Adoption_Rate_Upper"], marker="o", label="Upper Bound")
ax2.fill_between(wide["Year"], wide["Forecast_Adoption_P10"], wide["Forecast_Adoption_P90"], alpha=0.20, label="MC Forecast Band (P10–P90)")
ax2.plot(wide["Year"], wide["Forecast_Adoption_P50"], linestyle="--", linewidth=2, label="MC Median (P50)")
for (start, end, name), c in zip(phase_spans, phase_colors):
    ax2.axvspan(start, end, color=c, alpha=0.10)
line_handles, line_labels = ax2.get_legend_handles_labels()
phase_handles = [mpatches.Patch(color=c, alpha=0.10, label=name) for c, (_, _, name) in zip(phase_colors, phase_spans)]
ax2.legend(line_handles + phase_handles, line_labels + [name for *_, name in phase_spans], loc="best")
ax2.set_xlabel("Year")
ax2.set_ylabel("EV Adoption Rate")
ax2.set_title("King County: EV Adoption — Lower vs Upper vs Monotonic MC (Growth-Responsive)")
ax2.grid(True)
ax2.yaxis.set_major_formatter(PercentFormatter(1.0))  # treat 0–1 as 0–100%
plt.tight_layout()
adoption_chart = os.path.join(os.path.expanduser("~"), "Desktop", "ev_adoption_rate_mc_monotonic.png")
plt.savefig(adoption_chart)
plt.close()

# --- C) EV Registrations: MC P10/P50/P90 ---
plt.figure(figsize=(10, 6))
ax3 = plt.gca()
ax3.fill_between(wide["Year"], wide["Forecast_EVs_P10"], wide["Forecast_EVs_P90"], alpha=0.20, label="MC EVs Band (P10–P90)")
ax3.plot(wide["Year"], wide["Forecast_EVs_P50"], linestyle="--", linewidth=2, label="MC EVs Median (P50)")
for (start, end, name), c in zip(phase_spans, phase_colors):
    ax3.axvspan(start, end, color=c, alpha=0.10)
ax3.legend(loc="best")
ax3.set_xlabel("Year")
ax3.set_ylabel("EV Registrations (count)")
ax3.set_title("King County: EV Registrations — Monotonic MC P10 / P50 / P90")
ax3.grid(True)
plt.tight_layout()
evs_chart = os.path.join(os.path.expanduser("~"), "Desktop", "ev_registrations_mc_monotonic.png")
plt.savefig(evs_chart)
plt.close()

# =============================================================================
# Step 10: Embed charts back to Excel — PRESERVED
# =============================================================================
wb = load_workbook(desktop_path)
ws = wb.create_sheet(title="Charts")
ws.add_image(Image(charger_chart), "A1")
ws.add_image(Image(adoption_chart), "A40")
ws.add_image(Image(evs_chart), "A79")
wb.save(desktop_path)

# =============================================================================
# Final sanity checks & summary — PRESERVED
# =============================================================================
assert np.all(np.diff(wide["Forecast_Chargers"]) >= 0), "Final: chargers decreased — investigate!"
pct_within = 100.0 * wide["Forecast_within_bounds"].mean()
print(f"\n✅ Forecast complete. Excel and charts saved to: {desktop_path}")
print(f"• Median adoption within bounds in {pct_within:.1f}% of years.")
for yr in [2030, 2035, 2040, 2050]:
    r = wide.loc[wide['Year'] == yr].iloc[0]
    print(
        f" - {yr}: P50={r['Forecast_Adoption_P50']:.2%}, "
        f"Lower={r['Adoption_Rate_Lower']:.2%}, Upper={r['Adoption_Rate_Upper']:.2%}, "
        f"EVs_P50={int(r['Forecast_EVs_P50']):,}, Chargers={int(r['Forecast_Chargers']):,}"
    )


King County population (25-59): 1,241,805
Existing Superchargers: 13
Existing EVs (rows in EV file for King): 101838
Lower bound chargers (geo coverage): 184
Upper bound chargers (1 per 1,500 residents): 828

✅ Forecast complete. Excel and charts saved to: /Users/judycheng/Desktop/king_county_ev_projection_mc_monotonic.xlsx
• Median adoption within bounds in 80.8% of years.
 - 2030: P50=43.08%, Lower=22.25%, Upper=45.00%, EVs_P50=534,965, Chargers=297
 - 2035: P50=43.45%, Lower=35.62%, Upper=60.00%, EVs_P50=539,625, Chargers=297
 - 2040: P50=60.81%, Lower=49.00%, Upper=70.00%, EVs_P50=755,161, Chargers=300
 - 2050: P50=94.00%, Lower=95.00%, Upper=95.00%, EVs_P50=1,167,296, Chargers=538


In [ ]:
# -*- coding: utf-8 -*-
"""
Pierce County EV Adoption & Charger Forecast — Monotonic Monte Carlo
--------------------------------------------------------------------
✔ Keeps all graphs and Excel tabs identical
✔ Adds parameter block for all decimals with explanations
✔ Fully commented, ready to run
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, math
from openpyxl import load_workbook
from openpyxl.drawing.image import Image
from matplotlib.ticker import PercentFormatter
import matplotlib.patches as mpatches

# =============================================================================
# MODEL PARAMETERS  (all decimal constants centralized)
# =============================================================================

# --- Policy adoption targets (% of total population 25–59 adopting EVs) ---
POLICY_TARGETS = {2030: 0.45, 2035: 0.60, 2040: 0.70, 2050: 0.95}

# --- Charger density and area assumptions ---
RADIUS_MILES = 3.0           # each public charger covers ~3-mile radius
POP_PER_CHARGER = 2500.0     # 1 public charger per 2,500 residents (upper bound)
PIERCE_COUNTY_AREA = 1790    # sq mi

# --- Adoption baseline & tuning factors ---
BASE_2025_ADOPTION = 0.10    # baseline 10 % adoption in 2025
LOWER_PHASE1_MOD = 0.35      # 35 % of early growth toward 2030 target (moderated)
LOWER_PHASE2_MID = 0.70      # 70 % of target used for mid-phase plateau

# --- Upper-bound build-rate weights (front-load then taper) ---
UP_PHASE1_W = 1.0            # 2025–2030
UP_PHASE2_W = 0.6            # 2031–2040
UP_PHASE3_W = 0.2            # 2041–2050

# --- Monte Carlo adoption weighting ---
TARGET_CAP = 0.94            # adoption cap (< 95 %)
W_UPPER_P1 = 0.55            # blend weight toward upper path (2025–2030)
W_UPPER_P2 = 0.25            # lean lower early phase 2 (2031–2035)
W_UPPER_P3 = 0.70            # accelerate after 2035

# --- Charger blending for forecast path ---
W_CHARGER_P1 = 0.80          # early years emphasize upper build
W_CHARGER_P2 = 0.30          # transition to home-charging phase
W_CHARGER_P3 = 0.55          # moderate recovery post-2040

# --- Home-charger adoption trajectory (dampens public effect) ---
HOME_P1_START = 0.55
HOME_P1_END   = 0.72
HOME_P2_END   = 0.90
HOME_P3_END   = 0.93

# --- Charger growth & momentum ---
TRAIL_YEARS   = 3            # trailing window for avg momentum
MOMENTUM_CAP  = 0.40         # cap momentum ≤ 40 %
GROWTH_RATE_CAP = 0.50       # cap annual charger growth ≤ 50 %

# --- Monte Carlo phase parameters ---
MC_KAPPA = {"P1": 0.30, "P2": 0.25, "P3": 0.45, "P4": 0.55}  # mean-reversion to target
MC_BETA  = {"P1": 0.45, "P2": 0.20, "P3": 0.30, "P4": 0.30}  # charger-growth coupling
MC_SIGMA = {"P1": 0.015, "P2": 0.010, "P3": 0.018, "P4": 0.020}  # random noise
MC_BETA_MULT = 1.5           # amplify charger-momentum effect
MC_BETA_MAX  = 0.9           # stability clamp
MC_START_BLEND = (0.35, 0.65) # start adoption = 35 % Lower + 65 % Upper
MC_RUNS = 1000               # number of Monte Carlo paths

# --- Visualization phases ---
PHASE_SPANS  = [(2025,2030,"Phase 1"),(2030,2040,"Phase 2"),(2040,2050,"Phase 3")]
PHASE_COLORS = ["green","yellow","orange"]

# =============================================================================
# Step 1 – Paths (adjust if needed)
# =============================================================================
super_path     = "/Users/judycheng/Desktop/supercharger in washington state.xls"
residents_path = "/Users/judycheng/Desktop/Population 2024 age 25 to 59.xlsx"
ev_path        = "/Users/judycheng/Desktop/coordinates_output.xlsm"

# =============================================================================
# Step 2 – Load data for Pierce County
# =============================================================================
super_df     = pd.read_excel(super_path)
residents_df = pd.read_excel(residents_path)
ev_df        = pd.read_excel(ev_path)

pierce_super     = super_df[(super_df["State"]=="Washington") & (super_df["County"]=="Pierce County")]
pierce_residents = residents_df[residents_df["County"]=="Pierce County"]
pierce_ev        = ev_df[(ev_df["State"]=="WA") & (ev_df["County"]=="Pierce")]

population_25_59 = float(pierce_residents["total"].values[0])
num_chargers     = len(pierce_super)
num_evs          = len(pierce_ev)

print(f"Pierce County population (25–59): {population_25_59:,.0f}")
print(f"Existing Superchargers: {num_chargers}")
print(f"Existing EV records: {num_evs}")

# =============================================================================
# Step 3 – Policy Targets
# =============================================================================
policy_targets = POLICY_TARGETS

# =============================================================================
# Step 4 – Compute Charger Targets
# =============================================================================
area_per_charger = np.pi * RADIUS_MILES**2
lower_chargers   = int(np.ceil(PIERCE_COUNTY_AREA / area_per_charger))
upper_base_target = population_25_59 / POP_PER_CHARGER
print(f"Lower bound chargers (geo 3 mi): {lower_chargers}")
print(f"Upper bound (1 per 2,500 res): {upper_base_target:.0f}")

# =============================================================================
# Step 5 – Build Lower / Upper Scenarios
# =============================================================================
years = list(range(2025,2051))
results=[]
n_years=len(years)

# ---- Lower scenario ----
current_lower=num_chargers
total_lower_needed=max(0,lower_chargers-num_chargers)
yearly_build_lower_exact=total_lower_needed/n_years

for i,y in enumerate(years):
    new_lower=math.ceil(yearly_build_lower_exact)
    if current_lower+new_lower>lower_chargers or i==len(years)-1:
        new_lower=lower_chargers-current_lower
    new_lower=max(0,new_lower)
    current_lower+=new_lower

    if y<=2030:
        eff=BASE_2025_ADOPTION+(policy_targets[2030]-BASE_2025_ADOPTION)*((y-2025)/5)*LOWER_PHASE1_MOD
    elif y<=2040:
        eff_start=BASE_2025_ADOPTION+(policy_targets[2030]-BASE_2025_ADOPTION)*LOWER_PHASE1_MOD
        eff=eff_start+(policy_targets[2040]*LOWER_PHASE2_MID-eff_start)*((y-2030)/10)
    else:
        eff_start=policy_targets[2040]*LOWER_PHASE2_MID
        eff=eff_start+(policy_targets[2050]-eff_start)*((y-2040)/10)

    results.append({
        "Scenario":"Lower","Year":y,
        "Total_Chargers":int(current_lower),
        "New_Chargers":int(new_lower),
        "Adoption_Rate":float(np.clip(eff,0.0,0.9999))
    })

# ---- Upper scenario ----
base_build_rate=upper_base_target/n_years
current_upper=num_chargers

for y in years:
    if 2025<=y<=2030: yearly_add=int(round(base_build_rate*UP_PHASE1_W))
    elif 2031<=y<=2040: yearly_add=int(round(base_build_rate*UP_PHASE2_W))
    else: yearly_add=int(round(base_build_rate*UP_PHASE3_W))
    current_upper+=max(0,yearly_add)

    if y<=2030:
        eff=BASE_2025_ADOPTION+(policy_targets[2030]-BASE_2025_ADOPTION)*((y-2025)/5)
    elif y<=2035:
        eff=policy_targets[2030]+(policy_targets[2035]-policy_targets[2030])*((y-2030)/5)
    elif y<=2040:
        eff=policy_targets[2035]+(policy_targets[2040]-policy_targets[2035])*((y-2035)/5)
    else:
        eff=policy_targets[2040]+(policy_targets[2050]-policy_targets[2040])*((y-2040)/10)

    results.append({
        "Scenario":"Upper","Year":y,
        "Total_Chargers":int(current_upper),
        "New_Chargers":int(yearly_add),
        "Adoption_Rate":float(np.clip(eff,0.0,0.9999))
    })

forecast_df=pd.DataFrame(results)

# =============================================================================
# Step 6: Monte Carlo Simulation (monotone + charger momentum)
# -----------------------------------------------------------------------------
# This section simulates 1,000 possible EV adoption trajectories (Monte Carlo).
# For each simulated "path," we randomize small yearly deviations (σ) while
# anchoring adoption to policy targets and charger growth.
#
# After simulation:
#   - P10 = 10th percentile (conservative / slower adoption)
#   - P50 = 50th percentile (median / most likely adoption)
#   - P90 = 90th percentile (optimistic / faster adoption)
#
# These are used to create the shaded P10–P90 band and the median (P50) curve
# in the charts — visually showing uncertainty in EV adoption forecasts.
# =============================================================================

def clamp(x,lo=0.0,hi=0.9999): return float(np.clip(x,lo,hi))

yrs=np.array(years)
lo_path=forecast_df.pivot(index="Year",columns="Scenario",values="Adoption_Rate")["Lower"].values
hi_path=forecast_df.pivot(index="Year",columns="Scenario",values="Adoption_Rate")["Upper"].values

# --- Target path blend ---
target_w_upper=np.array([
    (W_UPPER_P1 if 2025<=y<=2030 else (W_UPPER_P2 if 2031<=y<=2035 else W_UPPER_P3))
    for y in yrs],dtype=float)
raw_target=(1-target_w_upper)*lo_path+target_w_upper*hi_path
target=np.minimum(raw_target,TARGET_CAP)
target=np.maximum.accumulate(target)

# --- Charger blend & growth ---
w_upper_charger=np.array([
    (W_CHARGER_P1 if 2025<=y<=2030 else (W_CHARGER_P2 if 2031<=y<=2040 else W_CHARGER_P3))
    for y in yrs],dtype=float)
lowC=forecast_df.pivot(index="Year",columns="Scenario",values="Total_Chargers")["Lower"].values
hiC =forecast_df.pivot(index="Year",columns="Scenario",values="Total_Chargers")["Upper"].values
C_blend=((1-w_upper_charger)*lowC+w_upper_charger*hiC).round().astype(int)
C_blend=np.maximum.accumulate(C_blend)
C_growth=np.r_[0,np.diff(C_blend)]
den=np.maximum(1.0,np.r_[C_blend[0],C_blend[:-1]])
C_growth_rate=np.clip(np.divide(C_growth,den,out=np.zeros_like(C_growth,dtype=float),where=den>0),0,GROWTH_RATE_CAP)

# --- Home-charger share & momentum ---
home_share=[]
for y in yrs:
    if 2025<=y<=2030:
        home_share.append(HOME_P1_START+(HOME_P1_END-HOME_P1_START)*(y-2025)/5)
    elif 2031<=y<=2040:
        home_share.append(HOME_P1_END+(HOME_P2_END-HOME_P1_END)*(y-2030)/10)
    else:
        home_share.append(HOME_P2_END+(HOME_P3_END-HOME_P2_END)*(y-2040)/10)
home_share=np.array(home_share)
pad=np.full(TRAIL_YEARS-1,C_growth_rate[1] if len(C_growth_rate)>1 else 0.0)
C_trailing=np.convolve(np.r_[pad,C_growth_rate],np.ones(TRAIL_YEARS)/TRAIL_YEARS,mode="valid")[:len(C_growth_rate)]
charger_momentum=np.clip(C_trailing,0.0,MOMENTUM_CAP)

# --- MC dynamics ---
N=MC_RUNS;T=len(yrs);rng=np.random.default_rng(42)
kappa_base=np.zeros(T);beta_base=np.zeros(T);sigma=np.zeros(T)
for i,y in enumerate(yrs):
    if 2025<=y<=2030: kappa_base[i]=MC_KAPPA["P1"];beta_base[i]=MC_BETA["P1"];sigma[i]=MC_SIGMA["P1"]
    elif 2031<=y<=2035: kappa_base[i]=MC_KAPPA["P2"];beta_base[i]=MC_BETA["P2"];sigma[i]=MC_SIGMA["P2"]
    elif 2036<=y<=2040: kappa_base[i]=MC_KAPPA["P3"];beta_base[i]=MC_BETA["P3"];sigma[i]=MC_SIGMA["P3"]
    else:               kappa_base[i]=MC_KAPPA["P4"];beta_base[i]=MC_BETA["P4"];sigma[i]=MC_SIGMA["P4"]

beta_effect=beta_base*(1-home_share)*(1+MC_BETA_MULT*charger_momentum)
beta_effect=np.clip(beta_effect,0.0,MC_BETA_MAX)

paths=np.zeros((N,T))
start_level=clamp(MC_START_BLEND[0]*lo_path[0]+MC_START_BLEND[1]*hi_path[0],0.0,TARGET_CAP)
paths[:,0]=start_level
for t in range(1,T):
    prev=paths[:,t-1]
    mean_revert=kappa_base[t]*(target[t]-prev)
    charger_push=beta_effect[t]*C_growth_rate[t]
    noise=rng.normal(0.0,sigma[t],size=N)
    delta=np.maximum(mean_revert+charger_push+noise,0.0)
    next_rate=np.minimum(np.maximum(prev+delta,lo_path[t]),TARGET_CAP)
    next_rate=np.maximum(next_rate,prev)
    paths[:,t]=next_rate

p10=np.percentile(paths,10,axis=0)
p50=np.percentile(paths,50,axis=0)
p90=np.percentile(paths,90,axis=0)
ev_p10=(p10*population_25_59).astype(int)
ev_p50=(p50*population_25_59).astype(int)
ev_p90=(p90*population_25_59).astype(int)

# =============================================================================
# Step 7 – Final Table
# =============================================================================
wide=forecast_df.pivot(index="Year",columns="Scenario",values=["Total_Chargers","Adoption_Rate"])
wide.columns=[f"{a}_{b}" for a,b in wide.columns];wide=wide.reset_index()
wide["Forecast_Chargers"]=C_blend
wide["Forecast_Adoption_P10"]=p10;wide["Forecast_Adoption_P50"]=p50;wide["Forecast_Adoption_P90"]=p90
wide["Forecast_EVs_P10"]=ev_p10;wide["Forecast_EVs_P50"]=ev_p50;wide["Forecast_EVs_P90"]=ev_p90
within_band=(wide["Forecast_Adoption_P50"].between(wide["Adoption_Rate_Lower"],wide["Adoption_Rate_Upper"]))
wide["Forecast_within_bounds"]=within_band.astype(int)

# =============================================================================
# Step 8 – Save to Excel
# =============================================================================
desktop_path=os.path.join(os.path.expanduser("~"),"Desktop","pierce_county_ev_projection_mc_monotonic.xlsx")
with pd.ExcelWriter(desktop_path,engine="openpyxl") as writer:
    forecast_df.to_excel(writer,sheet_name="Scenarios_LU",index=False)
    wide.to_excel(writer,sheet_name="Forecast",index=False)

# =============================================================================
# Step 9 – Charts (phase-shaded)
# =============================================================================
phase_spans=PHASE_SPANS;phase_colors=PHASE_COLORS

# A) Chargers
plt.figure(figsize=(10,6))
ax1=plt.gca()
ax1.plot(wide["Year"],wide["Total_Chargers_Lower"],marker="o",label="Lower Bound")
ax1.plot(wide["Year"],wide["Total_Chargers_Upper"],marker="o",label="Upper Bound")
ax1.plot(wide["Year"],wide["Forecast_Chargers"],"--",linewidth=2,label="Forecast (Monotone)")
for (s,e,n),c in zip(phase_spans,phase_colors): ax1.axvspan(s,e,color=c,alpha=0.10)
ax1.legend(loc="best");ax1.set_xlabel("Year");ax1.set_ylabel("Total Chargers")
ax1.set_title("Pierce County — EV Chargers Forecast (Monotone)");ax1.grid(True)
plt.tight_layout();charger_chart=os.path.join(os.path.expanduser("~"),"Desktop","pierce_charger_projection_mc_monotonic.png")
plt.savefig(charger_chart);plt.close()

# B) Adoption Rate
plt.figure(figsize=(10,6))
ax2=plt.gca()
ax2.plot(wide["Year"],wide["Adoption_Rate_Lower"],marker="o",label="Lower Bound")
ax2.plot(wide["Year"],wide["Adoption_Rate_Upper"],marker="o",label="Upper Bound")
ax2.fill_between(wide["Year"],wide["Forecast_Adoption_P10"],wide["Forecast_Adoption_P90"],alpha=0.20,label="MC Band (P10–P90)")
ax2.plot(wide["Year"],wide["Forecast_Adoption_P50"],"--",linewidth=2,label="MC Median (P50)")
for (s,e,n),c in zip(phase_spans,phase_colors): ax2.axvspan(s,e,color=c,alpha=0.10)
ax2.legend(loc="best");ax2.set_xlabel("Year");ax2.set_ylabel("EV Adoption Rate")
ax2.yaxis.set_major_formatter(PercentFormatter(1.0))
ax2.set_title("Pierce County — EV Adoption Forecast (Monotonic MC)");ax2.grid(True)
plt.tight_layout();adoption_chart=os.path.join(os.path.expanduser("~"),"Desktop","pierce_ev_adoption_rate_mc_monotonic.png")
plt.savefig(adoption_chart);plt.close()

# C) EV Registrations
plt.figure(figsize=(10,6))
ax3=plt.gca()
ax3.fill_between(wide["Year"],wide["Forecast_EVs_P10"],wide["Forecast_EVs_P90"],alpha=0.20,label="MC EVs Band (P10–P90)")
ax3.plot(wide["Year"],wide["Forecast_EVs_P50"],"--",linewidth=2,label="MC Median (P50)")
for (s,e,n),c in zip(phase_spans,phase_colors): ax3.axvspan(s,e,color=c,alpha=0.10)
ax3.legend(loc="best");ax3.set_xlabel("Year");ax3.set_ylabel("EV Registrations (count)")
ax3.set_title("Pierce County — EV Registrations (Monotonic MC)");ax3.grid(True)
plt.tight_layout();evs_chart=os.path.join(os.path.expanduser("~"),"Desktop","pierce_ev_registrations_mc_monotonic.png")
plt.savefig(evs_chart);plt.close()

# =============================================================================
# Step 10 – Embed Charts into Excel
# =============================================================================
wb=load_workbook(desktop_path);ws=wb.create_sheet(title="Charts")
ws.add_image(Image(charger_chart),"A1");ws.add_image(Image(adoption_chart),"A40");ws.add_image(Image(evs_chart),"A79")
wb.save(desktop_path)

# =============================================================================
# Final Checks & Summary
# =============================================================================
assert np.all(np.diff(wide["Forecast_Chargers"])>=0),"Final chargers decreased — investigate!"
pct_within=100.0*wide["Forecast_within_bounds"].mean()
print(f"\n✅ Projection complete — saved to {desktop_path}")
print(f"• Median adoption within bounds in {pct_within:.1f}% of years.")
print(f"• 2050 chargers (Upper): {int(forecast_df[forecast_df['Scenario']=='Upper'].iloc[-1]['Total_Chargers'])}")
print(f"• 2050 adoption P50: {p50[-1]:.2%} (cap {TARGET_CAP:.0%})")


Pierce County population (25–59): 448,201
Existing Superchargers: 1
Existing EV records: 16277
Lower bound chargers (geo 3 mi): 64
Upper bound (1 per 2,500 res): 179

✅ Projection complete — saved to /Users/judycheng/Desktop/pierce_county_ev_projection_mc_monotonic.xlsx
• Median adoption within bounds in 80.8% of years.
• 2050 chargers (Upper): 93
• 2050 adoption P50: 94.00% (cap 94%)


In [7]:
# -*- coding: utf-8 -*-
"""
Kitsap County EV Adoption & Charger Forecast — Monotonic Monte Carlo
--------------------------------------------------------------------
✔ Preserves logic, 3 charts, and Excel embedding
✔ Centralizes every decimal constant in MODEL PARAMETERS
✔ Uses the same MC tuning constants as King & Pierce
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, math
from openpyxl import load_workbook
from openpyxl.drawing.image import Image
from matplotlib.ticker import PercentFormatter
import matplotlib.patches as mpatches

# =============================================================================
# MODEL PARAMETERS (all decimal constants with explanations)
# =============================================================================

# --- Policy adoption targets (share of population 25–59 that adopt EVs) ---
POLICY_TARGETS = {2030: 0.45, 2035: 0.60, 2040: 0.70, 2050: 0.95}  # e.g., 0.45 = 45% by 2030

# --- Charger density & geography (Kitsap specifics) ---
KITSAP_COUNTY_AREA = 566        # sq mi
RADIUS_MILES = 5.0              # each public charger covers ~5-mile radius
POP_PER_CHARGER = 2500.0        # upper-bound target: 1 charger per 2,500 residents

# --- Early adoption baseline & Lower-scenario tuning ---
BASE_2025_ADOPTION = 0.10       # 10% baseline adoption in 2025
LOWER_P1_MOD = 0.35             # 35% of early growth toward 2030 target (moderation)
LOWER_P2_MID = 0.70             # 70% of 2040 target as Phase-2 midpoint

# --- Upper-scenario build weights (front-load then taper) ---
UP_PHASE1_W = 1.0               # 2025–2030
UP_PHASE2_W = 0.6               # 2031–2040
UP_PHASE3_W = 0.2               # 2041–2050

# --- Monte Carlo adoption target blending ---
TARGET_CAP = 0.94               # keep MC below ~95% by 2050
W_UPPER_P1 = 0.55               # 2025–2030: slightly upper-leaning
W_UPPER_P2 = 0.25               # 2031–2035: lean low early Phase 2
W_UPPER_P3 = 0.70               # 2036–2050: accelerate

# --- Charger blending for forecast path ---
W_CHARGER_P1 = 0.80             # early years emphasize upper charger build
W_CHARGER_P2 = 0.30             # transition years (home-charging rising)
W_CHARGER_P3 = 0.55             # later years partial recovery to public

# --- Home-charger share trajectory (dampens public-network effect) ---
HOME_P1_START = 0.55            # 55% at 2025
HOME_P1_END   = 0.72            # 72% by 2030
HOME_P2_END   = 0.90            # 90% by 2040
HOME_P3_END   = 0.93            # 93% by 2050

# --- Charger growth & momentum controls ---
TRAIL_YEARS     = 3             # trailing average window
MOMENTUM_CAP    = 0.40          # cap charger momentum ≤ 40%
GROWTH_RATE_CAP = 0.50          # cap annual charger growth ≤ 50%

# --- Monte Carlo dynamics (same constants as King & Pierce) ---
# Phases: P1=2025–2030, P2=2031–2035, P3=2036–2040, P4=2041–2050
MC_KAPPA = {"P1": 0.30, "P2": 0.25, "P3": 0.45, "P4": 0.55}   # mean-reversion strength
MC_BETA  = {"P1": 0.45, "P2": 0.20, "P3": 0.30, "P4": 0.30}   # charger-growth coupling baseline
MC_SIGMA = {"P1": 0.015, "P2": 0.010, "P3": 0.018, "P4": 0.020}  # random noise (std dev)
MC_BETA_MULT = 1.5                 # momentum amplifier for beta: (1 + 1.5 * momentum)
MC_BETA_MAX  = 0.9                 # stability clamp for beta_effect
MC_START_BLEND = (0.35, 0.65)      # start level: 35% Lower + 65% Upper
MC_RUNS = 1000                     # number of Monte Carlo paths

# --- Phase shading for charts ---
PHASE_SPANS  = [(2025, 2030, "Phase 1"), (2030, 2040, "Phase 2"), (2040, 2050, "Phase 3")]
PHASE_COLORS = ["green", "yellow", "orange"]  # visualization only

# =============================================================================
# Step 1: Read Excel files
# =============================================================================
super_df = pd.read_excel("/Users/judycheng/Desktop/supercharger in washington state.xls")
residents_df = pd.read_excel("/Users/judycheng/Desktop/Population 2024 age 25 to 59.xlsx")
ev_df = pd.read_excel("/Users/judycheng/Desktop/coordinates_output.xlsm")

# =============================================================================
# Step 2: Kitsap County data
# =============================================================================
kitsap_super = super_df[(super_df["State"] == "Washington") & (super_df["County"] == "Kitsap County")]
kitsap_residents = residents_df[residents_df["County"] == "Kitsap County"]
kitsap_ev = ev_df[(ev_df["State"] == "WA") & (ev_df["County"] == "Kitsap")]

population_column = "total"
population_25_59 = float(kitsap_residents[population_column].values[0])
num_chargers = int(len(kitsap_super))
num_evs = int(len(kitsap_ev))

print(f"Kitsap County population (25–59): {population_25_59:,.0f}")
print(f"Existing Superchargers: {num_chargers}")
print(f"Existing EVs: {num_evs}")

# =============================================================================
# Step 3: Policy targets
# =============================================================================
policy_targets = POLICY_TARGETS  # keep original variable name below

# =============================================================================
# Step 4: Charger targets
# =============================================================================
area_per_charger = np.pi * RADIUS_MILES**2                 # πr² with RADIUS_MILES=5.0
lower_chargers   = int(np.ceil(KITSAP_COUNTY_AREA / area_per_charger))  # geo coverage target
upper_base_target = population_25_59 / POP_PER_CHARGER     # 1 charger per 2,500 residents

print(f"Lower bound chargers (5-mile coverage): {lower_chargers}")
print(f"Upper bound (1 per 2,500 residents): {upper_base_target:.0f}")

# =============================================================================
# Step 5: Build Lower/Upper scenarios
# =============================================================================
years = list(range(2025, 2051))
results = []
n_years = len(years)

# ---- LOWER ----
current_lower = num_chargers
total_lower_needed = max(0, lower_chargers - num_chargers)
yearly_build_lower_exact = total_lower_needed / n_years  # float; we ceil each year

for i, y in enumerate(years):
    new_lower = math.ceil(yearly_build_lower_exact)
    if current_lower + new_lower > lower_chargers or i == len(years) - 1:
        new_lower = lower_chargers - current_lower
    new_lower = max(0, new_lower)
    current_lower += new_lower

    # Conservative adoption path (lower scenario)
    if y <= 2030:
        eff = BASE_2025_ADOPTION + (policy_targets[2030] - BASE_2025_ADOPTION) * ((y - 2025) / 5) * LOWER_P1_MOD
    elif y <= 2040:
        eff_start = BASE_2025_ADOPTION + (policy_targets[2030] - BASE_2025_ADOPTION) * LOWER_P1_MOD
        eff = eff_start + (policy_targets[2040] * LOWER_P2_MID - eff_start) * ((y - 2030) / 10)
    else:
        eff_start = policy_targets[2040] * LOWER_P2_MID
        eff = eff_start + (policy_targets[2050] - eff_start) * ((y - 2040) / 10)

    results.append({
        "Scenario": "Lower",
        "Year": y,
        "Total_Chargers": int(current_lower),
        "New_Chargers": int(new_lower),
        "Adoption_Rate": float(np.clip(eff, 0.0, 0.9999))
    })

# ---- UPPER ----
base_build_rate = upper_base_target / n_years
current_upper = num_chargers

for y in years:
    if 2025 <= y <= 2030:
        yearly_add = int(round(base_build_rate * UP_PHASE1_W))
    elif 2031 <= y <= 2040:
        yearly_add = int(round(base_build_rate * UP_PHASE2_W))
    else:
        yearly_add = int(round(base_build_rate * UP_PHASE3_W))
    yearly_add = max(0, yearly_add)
    current_upper += yearly_add

    # Upper adoption tracks policy milestones
    if y <= 2030:
        eff = BASE_2025_ADOPTION + (policy_targets[2030] - BASE_2025_ADOPTION) * ((y - 2025) / 5)
    elif y <= 2035:
        eff = policy_targets[2030] + (policy_targets[2035] - policy_targets[2030]) * ((y - 2030) / 5)
    elif y <= 2040:
        eff = policy_targets[2035] + (policy_targets[2040] - policy_targets[2035]) * ((y - 2035) / 5)
    else:
        eff = policy_targets[2040] + (policy_targets[2050] - policy_targets[2040]) * ((y - 2040) / 10)

    results.append({
        "Scenario": "Upper",
        "Year": y,
        "Total_Chargers": int(current_upper),
        "New_Chargers": int(yearly_add),
        "Adoption_Rate": float(np.clip(eff, 0.0, 0.9999))
    })

forecast_df = pd.DataFrame(results)

# =============================================================================
# Step 6: Monte Carlo Simulation (monotone + charger momentum)
# -----------------------------------------------------------------------------
# This section simulates 1,000 possible EV adoption trajectories (Monte Carlo).
# For each simulated "path," we randomize small yearly deviations (σ) while
# anchoring adoption to policy targets and charger growth.
#
# After simulation:
#   - P10 = 10th percentile (conservative / slower adoption)
#   - P50 = 50th percentile (median / most likely adoption)
#   - P90 = 90th percentile (optimistic / faster adoption)
#
# These are used to create the shaded P10–P90 band and the median (P50) curve
# in the charts — visually showing uncertainty in EV adoption forecasts.
# =============================================================================

def clamp(x, lo=0.0, hi=0.9999):
    return float(np.clip(x, lo, hi))

yrs = np.array(years)
lo_path = forecast_df.pivot(index="Year", columns="Scenario", values="Adoption_Rate")["Lower"].values
hi_path = forecast_df.pivot(index="Year", columns="Scenario", values="Adoption_Rate")["Upper"].values

# 6A) Target path the MC reverts toward (phase-weighted blend)
target_w_upper = np.array([
    (W_UPPER_P1 if 2025 <= y <= 2030 else (W_UPPER_P2 if 2031 <= y <= 2035 else W_UPPER_P3))
    for y in yrs
], dtype=float)

raw_target = (1 - target_w_upper) * lo_path + target_w_upper * hi_path
target = np.minimum(raw_target, TARGET_CAP)       # cap at 94%
target = np.maximum.accumulate(target)            # ensure non-decreasing

# 6B) Chargers + home-charging effects
w_upper_charger = np.array([
    (W_CHARGER_P1 if 2025 <= y <= 2030 else (W_CHARGER_P2 if 2031 <= y <= 2040 else W_CHARGER_P3))
    for y in yrs
], dtype=float)

lowC = forecast_df.pivot(index="Year", columns="Scenario", values="Total_Chargers")["Lower"].values
hiC  = forecast_df.pivot(index="Year", columns="Scenario", values="Total_Chargers")["Upper"].values

# Blended chargers (enforce monotonicity)
C_blend = ((1 - w_upper_charger) * lowC + w_upper_charger * hiC).round().astype(int)
C_blend = np.maximum.accumulate(C_blend)
assert np.all(np.diff(C_blend) >= 0), "Chargers decreased!"

# Growth rate (safe division) and cap
C_growth = np.r_[0, np.diff(C_blend)].astype(float)
den = np.maximum(1.0, np.r_[C_blend[0], C_blend[:-1]]).astype(float)
C_growth_rate = np.divide(C_growth, den, out=np.zeros_like(C_growth, dtype=float), where=den > 0)
C_growth_rate = np.clip(C_growth_rate, 0, GROWTH_RATE_CAP)

# Home-charger share (piecewise linear)
home_share = np.zeros_like(yrs, dtype=float)
for i, y in enumerate(yrs):
    if 2025 <= y <= 2030:
        home_share[i] = HOME_P1_START + (HOME_P1_END - HOME_P1_START) * (y - 2025) / 5
    elif 2031 <= y <= 2040:
        home_share[i] = HOME_P1_END + (HOME_P2_END - HOME_P1_END) * (y - 2030) / 10
    else:
        home_share[i] = HOME_P2_END + (HOME_P3_END - HOME_P2_END) * (y - 2040) / 10

# Charger momentum: 3-year trailing avg (capped)
pad = np.full(TRAIL_YEARS - 1, C_growth_rate[1] if len(C_growth_rate) > 1 else 0.0)
C_trailing = np.convolve(np.r_[pad, C_growth_rate], np.ones(TRAIL_YEARS)/TRAIL_YEARS, mode="valid")[:len(C_growth_rate)]
charger_momentum = np.clip(C_trailing, 0, MOMENTUM_CAP)

# 6C) MC evolution (monotone)
N, T = MC_RUNS, len(yrs)
rng = np.random.default_rng(42)
kappa = np.zeros(T); beta = np.zeros(T); sigma = np.zeros(T)

for i, y in enumerate(yrs):
    if 2025 <= y <= 2030:
        kappa[i] = MC_KAPPA["P1"]; beta[i] = MC_BETA["P1"]; sigma[i] = MC_SIGMA["P1"]
    elif 2031 <= y <= 2035:
        kappa[i] = MC_KAPPA["P2"]; beta[i] = MC_BETA["P2"]; sigma[i] = MC_SIGMA["P2"]
    elif 2036 <= y <= 2040:
        kappa[i] = MC_KAPPA["P3"]; beta[i] = MC_BETA["P3"]; sigma[i] = MC_SIGMA["P3"]
    else:
        kappa[i] = MC_KAPPA["P4"]; beta[i] = MC_BETA["P4"]; sigma[i] = MC_SIGMA["P4"]

beta_effect = beta * (1 - home_share) * (1 + MC_BETA_MULT * charger_momentum)
beta_effect = np.clip(beta_effect, 0, MC_BETA_MAX)

paths = np.zeros((N, T))
start_level = clamp(MC_START_BLEND[0] * lo_path[0] + MC_START_BLEND[1] * hi_path[0], 0.0, TARGET_CAP)
paths[:, 0] = start_level

for t in range(1, T):
    prev = paths[:, t-1]
    mean_revert = kappa[t] * (target[t] - prev)
    charger_push = beta_effect[t] * C_growth_rate[t]
    noise = rng.normal(0.0, sigma[t], size=N)
    delta = np.maximum(mean_revert + charger_push + noise, 0.0)  # no negative increments
    next_rate = np.minimum(np.maximum(prev + delta, lo_path[t]), TARGET_CAP)
    next_rate = np.maximum(next_rate, prev)  # enforce monotonicity
    paths[:, t] = next_rate

# MC percentiles
p10 = np.percentile(paths, 10, axis=0)
p50 = np.percentile(paths, 50, axis=0)
p90 = np.percentile(paths, 90, axis=0)

ev_p10 = (p10 * population_25_59).astype(int)
ev_p50 = (p50 * population_25_59).astype(int)
ev_p90 = (p90 * population_25_59).astype(int)

# =============================================================================
# Step 7: Assemble + save
# =============================================================================
wide = forecast_df.pivot(index="Year", columns="Scenario", values=["Total_Chargers", "Adoption_Rate"])
wide.columns = [f"{a}_{b}" for a, b in wide.columns]
wide = wide.reset_index()

wide["Forecast_Adoption_P10"] = p10
wide["Forecast_Adoption_P50"] = p50
wide["Forecast_Adoption_P90"] = p90
wide["Forecast_EVs_P10"] = ev_p10
wide["Forecast_EVs_P50"] = ev_p50
wide["Forecast_EVs_P90"] = ev_p90
wide["Forecast_Chargers"] = C_blend

desktop_path = os.path.join(os.path.expanduser("~"), "Desktop", "kitsap_county_ev_projection_mc_monotonic.xlsx")
with pd.ExcelWriter(desktop_path, engine="openpyxl") as writer:
    forecast_df.to_excel(writer, sheet_name="Scenarios_LU", index=False)
    wide.to_excel(writer, sheet_name="Forecast", index=False)

# =============================================================================
# Step 8: Charts (phase-shaded)
# =============================================================================
phase_spans = PHASE_SPANS
phase_colors = PHASE_COLORS

# Chargers
plt.figure(figsize=(10, 6))
ax1 = plt.gca()
ax1.plot(wide["Year"], wide["Total_Chargers_Lower"], marker="o", label="Lower Bound")
ax1.plot(wide["Year"], wide["Total_Chargers_Upper"], marker="o", label="Upper Bound")
ax1.plot(wide["Year"], wide["Forecast_Chargers"], "--", linewidth=2, label="Forecast (Monotone)")
for (s, e, n), c in zip(phase_spans, phase_colors):
    ax1.axvspan(s, e, color=c, alpha=0.10)
ax1.legend(loc="best")
ax1.set_xlabel("Year")
ax1.set_ylabel("Total Chargers")
ax1.set_title("Kitsap County: EV Chargers — Lower / Upper / Monotone Forecast")
ax1.grid(True)
plt.tight_layout()
charger_chart = os.path.join(os.path.expanduser("~"), "Desktop", "kitsap_charger_projection_mc_monotonic.png")
plt.savefig(charger_chart)
plt.close()

# Adoption
plt.figure(figsize=(10, 6))
ax2 = plt.gca()
ax2.plot(wide["Year"], wide["Adoption_Rate_Lower"], marker="o", label="Lower Bound")
ax2.plot(wide["Year"], wide["Adoption_Rate_Upper"], marker="o", label="Upper Bound")
ax2.fill_between(wide["Year"], wide["Forecast_Adoption_P10"], wide["Forecast_Adoption_P90"], alpha=0.20, label="MC Band P10–P90")
ax2.plot(wide["Year"], wide["Forecast_Adoption_P50"], "--", linewidth=2, label="MC Median (P50)")
for (s, e, n), c in zip(phase_spans, phase_colors):
    ax2.axvspan(s, e, color=c, alpha=0.10)
ax2.legend(loc="best")
ax2.set_xlabel("Year")
ax2.set_ylabel("EV Adoption Rate")
ax2.set_title("Kitsap County: EV Adoption — Monotone MC (Growth-Responsive)")
ax2.grid(True)
ax2.yaxis.set_major_formatter(PercentFormatter(1.0))
plt.tight_layout()
adoption_chart = os.path.join(os.path.expanduser("~"), "Desktop", "kitsap_ev_adoption_rate_mc_monotonic.png")
plt.savefig(adoption_chart)
plt.close()

# Registrations
plt.figure(figsize=(10, 6))
ax3 = plt.gca()
ax3.fill_between(wide["Year"], wide["Forecast_EVs_P10"], wide["Forecast_EVs_P90"], alpha=0.20, label="MC EVs Band (P10–P90)")
ax3.plot(wide["Year"], wide["Forecast_EVs_P50"], "--", linewidth=2, label="MC EVs Median (P50)")
for (s, e, n), c in zip(phase_spans, phase_colors):
    ax3.axvspan(s, e, color=c, alpha=0.10)
ax3.legend(loc="best")
ax3.set_xlabel("Year")
ax3.set_ylabel("EV Registrations (count)")
ax3.set_title("Kitsap County: EV Registrations — Monotonic MC P10 / P50 / P90")
ax3.grid(True)
plt.tight_layout()
evs_chart = os.path.join(os.path.expanduser("~"), "Desktop", "kitsap_ev_registrations_mc_monotonic.png")
plt.savefig(evs_chart)
plt.close()

# =============================================================================
# Step 9: Embed charts in Excel
# =============================================================================
wb = load_workbook(desktop_path)
ws = wb.create_sheet(title="Charts")
ws.add_image(Image(charger_chart), "A1")
ws.add_image(Image(adoption_chart), "A40")
ws.add_image(Image(evs_chart), "A79")
wb.save(desktop_path)

# =============================================================================
# Final summary
# =============================================================================
assert np.all(np.diff(wide["Forecast_Chargers"]) >= 0), "Final forecast not monotonic!"
print(f"\n✅ Kitsap projection complete: {desktop_path}")
print(f"2050 P50 adoption: {p50[-1]:.2%} (capped <{int(TARGET_CAP*100)}%)")


Kitsap County population (25–59): 125,820
Existing Superchargers: 0
Existing EVs: 6428
Lower bound chargers (5-mile coverage): 8
Upper bound (1 per 2,500 residents): 50

✅ Kitsap projection complete: /Users/judycheng/Desktop/kitsap_county_ev_projection_mc_monotonic.xlsx
2050 P50 adoption: 94.00% (capped <94%)


In [ ]:
# -*- coding: utf-8 -*-
"""
Chelan County EV Adoption & Charger Forecast — Monotonic Monte Carlo
--------------------------------------------------------------------
✔ Preserves original logic, outputs, 3 charts, and Excel embedding
✔ Centralizes every decimal constant in MODEL PARAMETERS
✔ Uses same MC tuning style as your King/Pierce/Kitsap scripts
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, math
from openpyxl import load_workbook
from openpyxl.drawing.image import Image
import matplotlib.patches as mpatches
from matplotlib.ticker import PercentFormatter

# =============================================================================
# MODEL PARAMETERS (all decimal constants with concise explanations)
# =============================================================================

# --- Policy adoption targets (share of population 25–59 that adopt EVs) ---
POLICY_TARGETS = {2030: 0.45, 2035: 0.60, 2040: 0.70, 2050: 0.95}  # e.g., 0.45 = 45% by 2030

# --- Chelan geography & charger density assumptions ---
COUNTY_AREA_SQMI = 2965         # Chelan County area in square miles
RADIUS_MILES     = 15.0          # each charger covers ~15-mile radius (rural coverage)
POP_PER_CHARGER  = 5000.0        # upper-bound target: 1 public charger per 5,000 residents

# --- Lower scenario tuning & early baseline adoption ---
BASE_2025_ADOPTION = 0.10        # 10% baseline adoption at 2025
LOWER_P1_MOD       = 0.35        # 35% of early growth toward 2030 target (moderation)
LOWER_P2_MID       = 0.70        # 70% of 2040 target used as Phase-2 midpoint plateau

# --- Monte Carlo adoption target blending (between Lower/Upper paths) ---
TARGET_CAP  = 0.94               # cap MC < 95% by 2050
W_UPPER_P1  = 0.55               # 2025–2030 slight upper-lean
W_UPPER_P2  = 0.25               # 2031–2035 lean lower
W_UPPER_P3  = 0.70               # 2036–2050 accelerate toward upper

# --- Charger blending for forecast path (between Lower/Upper chargers) ---
W_CHARGER_P1 = 0.80              # early years emphasize upper charger build
W_CHARGER_P2 = 0.30              # transition years due to rising home charging
W_CHARGER_P3 = 0.55              # later years partial recovery for public network

# --- Home-charger share trajectory (rural tilt) ---
HOME_P1_START = 0.50             # 50% at 2025
HOME_P1_END   = 0.70             # to 70% by 2030
HOME_P2_END   = 0.88             # to 88% by 2040
HOME_P3_END   = 0.92             # to 92% by 2050

# --- Charger growth & momentum controls ---
TRAIL_YEARS      = 3             # trailing window for momentum average
MOMENTUM_CAP     = 0.40          # cap charger momentum to 40%
GROWTH_RATE_CAP  = 0.50          # cap annual charger growth rate to 50%

# --- Monte Carlo dynamics (same structure/values as other counties) ---
# Phases: P1=2025–2030, P2=2031–2035, P3=2036–2040, P4=2041–2050
MC_KAPPA = {"P1": 0.30, "P2": 0.25, "P3": 0.45, "P4": 0.55}  # mean-reversion strength to target
MC_BETA  = {"P1": 0.45, "P2": 0.20, "P3": 0.30, "P4": 0.30}  # charger-growth coupling baseline
MC_SIGMA = {"P1": 0.015, "P2": 0.010, "P3": 0.018, "P4": 0.020}  # noise std by phase
MC_BETA_MULT = 1.5               # beta amplified by (1 + 1.5 * momentum)
MC_BETA_MAX  = 0.90              # clamp effective beta for stability
MC_START_BLEND = (0.35, 0.65)    # start level = 35% Lower + 65% Upper
MC_RUNS = 1000                   # Monte Carlo paths

# --- Chart phase shading (visual only) ---
PHASE_SPANS  = [(2025, 2030, "Phase 1"), (2030, 2040, "Phase 2"), (2040, 2050, "Phase 3")]
PHASE_COLORS = ["green", "yellow", "orange"]

# =============================================================================
# Step 1: Read Excel files
# =============================================================================
super_df = pd.read_excel("/Users/judycheng/Desktop/supercharger in washington state.xls")
residents_df = pd.read_excel("/Users/judycheng/Desktop/Population 2024 age 25 to 59.xlsx")
ev_df = pd.read_excel("/Users/judycheng/Desktop/coordinates_output.xlsm")

# =============================================================================
# Step 2: Chelan County data
# =============================================================================
chelan_super = super_df[(super_df["State"] == "Washington") & (super_df["County"] == "Chelan County")]
chelan_residents = residents_df[residents_df["County"] == "Chelan County"]
chelan_ev = ev_df[(ev_df["State"] == "WA") & (ev_df["County"] == "Chelan")]

population_column = "total"
population_25_59 = float(chelan_residents[population_column].values[0])
num_chargers = int(len(chelan_super))
num_evs = int(len(chelan_ev))

print(f"Chelan County population (25–59): {population_25_59:,.0f}")
print(f"Existing Superchargers: {num_chargers}")
print(f"Existing EVs: {num_evs}")

# =============================================================================
# Step 3: Policy targets
# =============================================================================
policy_targets = POLICY_TARGETS  # keep variable name used later

# =============================================================================
# Step 4: Charger targets
# =============================================================================
area_per_charger = np.pi * RADIUS_MILES**2  # π * r^2 using 15-mile radius
lower_chargers = math.ceil(COUNTY_AREA_SQMI / area_per_charger)   # ~15-mi geo coverage
upper_chargers = math.ceil(population_25_59 / POP_PER_CHARGER)    # 1 per 5,000 residents

print(f"Lower bound chargers (15-mi coverage): {lower_chargers}")
print(f"Upper bound chargers (1 per 5,000 residents): {upper_chargers}")

# =============================================================================
# Step 5: Build Lower / Upper scenarios
# =============================================================================
years = list(range(2025, 2051))
results = []

# ---- LOWER: steady build toward geo-coverage; conservative adoption in Phase 2 ----
current_lower = num_chargers
total_lower_needed = max(0, lower_chargers - num_chargers)
n_years = len(years)
yearly_build_lower = int(round(total_lower_needed / n_years)) if n_years > 0 else 0
yearly_build_lower = max(1, yearly_build_lower) if total_lower_needed > 0 else 0  # ensure progress if needed

for i, y in enumerate(years):
    new_lower = yearly_build_lower if current_lower < lower_chargers else 0
    if current_lower + new_lower > lower_chargers:
        new_lower = lower_chargers - current_lower
    current_lower += max(0, new_lower)

    # Lower adoption path
    if y <= 2030:
        eff = BASE_2025_ADOPTION + (policy_targets[2030] - BASE_2025_ADOPTION) * ((y - 2025) / 5) * LOWER_P1_MOD
    elif y <= 2040:
        eff_start = BASE_2025_ADOPTION + (policy_targets[2030] - BASE_2025_ADOPTION) * LOWER_P1_MOD
        eff = eff_start + (policy_targets[2040] * LOWER_P2_MID - eff_start) * ((y - 2030) / 10)
    else:
        eff_start = policy_targets[2040] * LOWER_P2_MID
        eff = eff_start + (policy_targets[2050] - eff_start) * ((y - 2040) / 10)

    results.append({
        "Scenario":"Lower","Year":y,
        "Total_Chargers":int(current_lower),
        "New_Chargers":int(max(0,new_lower)),
        "Adoption_Rate":float(np.clip(eff,0.0,0.9999)),
        "EVs":int(population_25_59 * np.clip(eff,0.0,0.9999))
    })

# ---- UPPER: smooth path to population target; adoption follows policy milestones ----
current_upper = num_chargers
for y in years:
    remaining = max(0, upper_chargers - current_upper)
    remaining_years = max(1, 2050 - y + 1)
    yearly_add = int(math.ceil(remaining / remaining_years)) if remaining > 0 else 0
    current_upper += yearly_add

    if y <= 2030:
        eff = BASE_2025_ADOPTION + (policy_targets[2030]-BASE_2025_ADOPTION) * ((y-2025)/5)
    elif y <= 2035:
        eff = policy_targets[2030] + (policy_targets[2035]-policy_targets[2030]) * ((y-2030)/5)
    elif y <= 2040:
        eff = policy_targets[2035] + (policy_targets[2040]-policy_targets[2035]) * ((y-2035)/5)
    else:
        eff = policy_targets[2040] + (policy_targets[2050]-policy_targets[2040]) * ((y-2040)/10)

    results.append({
        "Scenario":"Upper","Year":y,
        "Total_Chargers":int(current_upper),
        "New_Chargers":int(yearly_add),
        "Adoption_Rate":float(np.clip(eff,0.0,0.9999)),
        "EVs":int(population_25_59 * np.clip(eff,0.0,0.9999))
    })

sc_df = pd.DataFrame(results)

# =============================================================================
# Step 6: Monte Carlo Simulation (monotone + charger momentum)
# -----------------------------------------------------------------------------
# This section simulates 1,000 possible EV adoption trajectories (Monte Carlo).
# For each simulated "path," we randomize small yearly deviations (σ) while
# anchoring adoption to policy targets and charger growth.
#
# After simulation:
#   - P10 = 10th percentile (conservative / slower adoption)
#   - P50 = 50th percentile (median / most likely adoption)
#   - P90 = 90th percentile (optimistic / faster adoption)
#
# These are used to create the shaded P10–P90 band and the median (P50) curve
# in the charts — visually showing uncertainty in EV adoption forecasts.
# =============================================================================

def clamp(x, lo=0.0, hi=0.9999):
    return float(np.clip(x, lo, hi))

yrs = sc_df["Year"].unique()
lo_path = sc_df.pivot(index="Year", columns="Scenario", values="Adoption_Rate")["Lower"].values
hi_path = sc_df.pivot(index="Year", columns="Scenario", values="Adoption_Rate")["Upper"].values

# 6A) Target path (not midpoint): weights vary by phase; cap < 95%
target_w_upper = np.array([
    (W_UPPER_P1 if 2025 <= y <= 2030 else (W_UPPER_P2 if 2031 <= y <= 2035 else W_UPPER_P3))
    for y in yrs
], dtype=float)

raw_target = (1 - target_w_upper) * lo_path + target_w_upper * hi_path
target = np.minimum(raw_target, TARGET_CAP)
target = np.maximum.accumulate(target)  # ensure non-decreasing target

# 6B) Charger blend (monotone), growth rate (float-safe), home-share, momentum
w_upper_charger = np.array([
    (W_CHARGER_P1 if 2025 <= y <= 2030 else (W_CHARGER_P2 if 2031 <= y <= 2040 else W_CHARGER_P3))
    for y in yrs
], dtype=float)

lowC = sc_df.pivot(index="Year", columns="Scenario", values="Total_Chargers")["Lower"].values
hiC  = sc_df.pivot(index="Year", columns="Scenario", values="Total_Chargers")["Upper"].values

# ✅ Enforce monotonic chargers for the blended "forecast" path
C_blend = ((1 - w_upper_charger) * lowC + w_upper_charger * hiC).round().astype(int)
C_blend = np.maximum.accumulate(C_blend)
assert np.all(np.diff(C_blend) >= 0), "Internal: blended charger path decreased!"

# Float-safe charger growth rate
C_growth = np.r_[0, np.diff(C_blend)].astype(float)
den = np.maximum(1.0, np.r_[C_blend[0], C_blend[:-1]]).astype(float)
C_growth_rate = np.divide(C_growth, den, out=np.zeros_like(C_growth, dtype=float), where=den > 0)
C_growth_rate = np.clip(C_growth_rate, 0, GROWTH_RATE_CAP)

# Home-charging share (rural progression 50%→70%→88%→92%)
home_share = np.zeros_like(yrs, dtype=float)
for i, y in enumerate(yrs):
    if 2025 <= y <= 2030:
        home_share[i] = HOME_P1_START + (HOME_P1_END - HOME_P1_START) * (y - 2025) / 5
    elif 2031 <= y <= 2040:
        home_share[i] = HOME_P1_END + (HOME_P2_END - HOME_P1_END) * (y - 2030) / 10
    else:
        home_share[i] = HOME_P2_END + (HOME_P3_END - HOME_P2_END) * (y - 2040) / 10

# Charger momentum: 3-year trailing average of growth
pad = np.full(TRAIL_YEARS - 1, C_growth_rate[1] if len(C_growth_rate) > 1 else 0.0)
C_gr_trailing = np.convolve(np.r_[pad, C_growth_rate], np.ones(TRAIL_YEARS)/TRAIL_YEARS, mode="valid")[:len(C_growth_rate)]
charger_momentum = np.clip(C_gr_trailing, 0.0, MOMENTUM_CAP)

# 6C) MC evolution (monotone)
N = MC_RUNS
T = len(yrs)
rng = np.random.default_rng(42)

kappa = np.zeros(T); beta = np.zeros(T); sigma = np.zeros(T)
for i, y in enumerate(yrs):
    if 2025 <= y <= 2030:
        kappa[i]=MC_KAPPA["P1"]; beta[i]=MC_BETA["P1"]; sigma[i]=MC_SIGMA["P1"]
    elif 2031 <= y <= 2035:
        kappa[i]=MC_KAPPA["P2"]; beta[i]=MC_BETA["P2"]; sigma[i]=MC_SIGMA["P2"]
    elif 2036 <= y <= 2040:
        kappa[i]=MC_KAPPA["P3"]; beta[i]=MC_BETA["P3"]; sigma[i]=MC_SIGMA["P3"]
    else:
        kappa[i]=MC_KAPPA["P4"]; beta[i]=MC_BETA["P4"]; sigma[i]=MC_SIGMA["P4"]

# Couple adoption to charger growth, damped by home_share and boosted by momentum
beta_effect = beta * (1 - home_share) * (1 + MC_BETA_MULT * charger_momentum)
beta_effect = np.clip(beta_effect, 0.0, MC_BETA_MAX)

paths = np.zeros((N, T), dtype=float)
start_level = clamp(MC_START_BLEND[0] * lo_path[0] + MC_START_BLEND[1] * hi_path[0], 0.0, TARGET_CAP)  # rural corridor uplift early
paths[:, 0] = start_level

for t in range(1, T):
    prev = paths[:, t-1]
    mean_revert = kappa[t] * (target[t] - prev)
    charger_push = beta_effect[t] * C_growth_rate[t]
    noise = rng.normal(0.0, sigma[t], size=N)
    delta = np.maximum(mean_revert + charger_push + noise, 0.0)   # forbid negative increments
    next_rate = prev + delta
    next_rate = np.maximum(next_rate, lo_path[t])                  # never below Lower path that year
    next_rate = np.minimum(next_rate, TARGET_CAP)                  # keep < 95%
    next_rate = np.maximum(next_rate, prev)                        # enforce monotonicity
    paths[:, t] = next_rate

# MC percentiles and EV counts
p10 = np.percentile(paths, 10, axis=0)
p50 = np.percentile(paths, 50, axis=0)
p90 = np.percentile(paths, 90, axis=0)
ev_p10 = (p10 * population_25_59).astype(int)
ev_p50 = (p50 * population_25_59).astype(int)
ev_p90 = (p90 * population_25_59).astype(int)

# =============================================================================
# Step 7: Save results with forecast columns
# =============================================================================
wide = sc_df.pivot(index="Year", columns="Scenario", values=["Total_Chargers", "Adoption_Rate"])
wide.columns = [f"{a}_{b}" for a, b in wide.columns]
wide = wide.reset_index()

wide["Forecast_Adoption_P10"] = p10
wide["Forecast_Adoption_P50"] = p50
wide["Forecast_Adoption_P90"] = p90
wide["Forecast_EVs_P10"] = ev_p10
wide["Forecast_EVs_P50"] = ev_p50
wide["Forecast_EVs_P90"] = ev_p90

# ✅ Use the monotone blended charger path as the forecast
wide["Forecast_Chargers"] = C_blend
assert np.all(np.diff(wide["Forecast_Chargers"]) >= 0), "Final: charger forecast not monotone!"

desktop_path = os.path.join(os.path.expanduser("~"), "Desktop", "chelan_county_ev_projection_mc_monotonic.xlsx")
with pd.ExcelWriter(desktop_path, engine="openpyxl") as writer:
    sc_df.to_excel(writer, sheet_name="Scenarios_LU", index=False)
    wide.to_excel(writer, sheet_name="Forecast", index=False)

# =============================================================================
# Step 8: Charts (with phase shading)
# =============================================================================
phase_spans = PHASE_SPANS
phase_colors = PHASE_COLORS

# 1) Chargers
plt.figure(figsize=(10,6))
ax1 = plt.gca()
ax1.plot(wide["Year"], wide["Total_Chargers_Lower"], marker="o", label="Lower Bound")
ax1.plot(wide["Year"], wide["Total_Chargers_Upper"], marker="o", label="Upper Bound")
ax1.plot(wide["Year"], wide["Forecast_Chargers"], "--", linewidth=2, label="Forecast (Monotone)")
for (s,e,n),c in zip(phase_spans,phase_colors): ax1.axvspan(s,e,color=c,alpha=0.10)
ax1.legend(loc="best")
ax1.set_xlabel("Year"); ax1.set_ylabel("Total Chargers")
ax1.set_title("Chelan County: EV Chargers — Lower / Upper / Monotone Forecast")
ax1.grid(True)
plt.tight_layout()
charger_chart = os.path.join(os.path.expanduser("~"), "Desktop", "chelan_chargers_mc_monotonic.png")
plt.savefig(charger_chart); plt.close()

# 2) Adoption Rate
plt.figure(figsize=(10,6))
ax2 = plt.gca()
ax2.plot(wide["Year"], wide["Adoption_Rate_Lower"], marker="o", label="Lower Bound")
ax2.plot(wide["Year"], wide["Adoption_Rate_Upper"], marker="o", label="Upper Bound")
ax2.fill_between(wide["Year"], wide["Forecast_Adoption_P10"], wide["Forecast_Adoption_P90"], alpha=0.20, label="MC Band (P10–P90)")
ax2.plot(wide["Year"], wide["Forecast_Adoption_P50"], "--", linewidth=2, label="MC Median (P50)")
for (s,e,n),c in zip(phase_spans,phase_colors): ax2.axvspan(s,e,color=c,alpha=0.10)
ax2.legend(loc="best")
ax2.set_xlabel("Year"); ax2.set_ylabel("EV Adoption Rate")
ax2.set_title("Chelan County: EV Adoption — Monotone MC (Growth-Responsive)")
ax2.grid(True); ax2.yaxis.set_major_formatter(PercentFormatter(1.0))
plt.tight_layout()
adoption_chart = os.path.join(os.path.expanduser("~"), "Desktop", "chelan_adoption_mc_monotonic.png")
plt.savefig(adoption_chart); plt.close()

# 3) EV Registrations
plt.figure(figsize=(10,6))
ax3 = plt.gca()
ax3.fill_between(wide["Year"], wide["Forecast_EVs_P10"], wide["Forecast_EVs_P90"], alpha=0.20, label="MC EVs Band (P10–P90)")
ax3.plot(wide["Year"], wide["Forecast_EVs_P50"], "--", linewidth=2, label="MC EVs Median (P50)")
for (s,e,n),c in zip(phase_spans,phase_colors): ax3.axvspan(s,e,color=c,alpha=0.10)
ax3.legend(loc="best")
ax3.set_xlabel("Year"); ax3.set_ylabel("EV Registrations (count)")
ax3.set_title("Chelan County: EV Registrations — Monotone MC P10 / P50 / P90")
ax3.grid(True)
plt.tight_layout()
evs_chart = os.path.join(os.path.expanduser("~"), "Desktop", "chelan_evs_mc_monotonic.png")
plt.savefig(evs_chart); plt.close()

# =============================================================================
# Step 9: Embed charts in Excel
# =============================================================================
wb = load_workbook(desktop_path)
ws = wb.create_sheet(title="Charts")
ws.add_image(Image(charger_chart), "A1")
ws.add_image(Image(adoption_chart), "A40")
ws.add_image(Image(evs_chart), "A79")
wb.save(desktop_path)

# =============================================================================
# Summary
# =============================================================================
print(f"\n✅ Chelan County EV Projection saved to: {desktop_path}")
print(f"2050 P50 adoption (capped <{int(TARGET_CAP*100)}%): {p50[-1]:.2%}")


Chelan County population (25–59): 33,879
Existing Superchargers: 3
Existing EVs: 1250
Lower bound chargers (15-mi coverage): 5
Upper bound chargers (1 per 5,000 residents): 7

✅ Chelan County EV Projection saved to: /Users/judycheng/Desktop/chelan_county_ev_projection_mc_monotonic.xlsx
2050 P50 adoption (capped <94%): 94.00%
